In [1]:
!pip install fairlearn pgmpy optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

CACHE_DIR = '/kaggle/working/'
os.makedirs(CACHE_DIR, exist_ok=True)

def clean_numeric_column(series):
    series = pd.to_numeric(series, errors='coerce')
    series = series.replace([np.inf, -np.inf], np.nan)
    series = series.fillna(series.median())
    return series

def safe_qcut(series, q=5):
    series = clean_numeric_column(series)
    if series.nunique() <= 1:
        return pd.Series(0, index=series.index, dtype=int)
    try:
        return pd.qcut(series, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except:
        try:
            return pd.cut(series, q, labels=False, duplicates='drop').fillna(0).astype(int)
        except:
            return pd.Series(0, index=series.index, dtype=int)

def preprocess_adult_for_fair_bbn(csv_path='/kaggle/input/all-datasets/adult.csv', use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_adult.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Adult data...")
        return joblib.load(cache_file)
    
    df = pd.read_csv(csv_path)
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True)
    df['income'] = np.where(df['income']=='>50K', 1, 0)
    
    race_map = {"White": 0,"Black": 1,"Asian-Pac-Islander": 2,"Amer-Indian-Eskimo": 3,"Other": 4}
    sex_map = {"Male": 0,"Female": 1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    
    categorical_cols = ['workclass','education','marital.status','occupation','relationship','native.country']
    numeric_cols = ['age','fnlwgt','education.num','capital.gain','capital.loss','hours.per.week']
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['income','race_label','sex_label'])
    y = df['income'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Adult data to {cache_file}")
    
    return result

def preprocess_compas_for_fair_bbn(csv_path='/kaggle/input/all-datasets/compas-scores-two-years.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_compas.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached COMPAS data...")
        return joblib.load(cache_file)
    
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'] <= 30) &
        (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') &
        (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex',
         'priors_count','days_b_screening_arrest','decile_score',
         'juv_other_count','juv_misd_count','juv_fel_count',
         'c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)
    
    race_map = {"African-American":0,"Caucasian":1,"Hispanic":2,"Other":3,"Asian":4,"Native American":5}
    sex_map = {"Male":0,"Female":1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    
    df['c_jail_in'] = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time'] = (df['c_jail_out'] - df['c_jail_in']).dt.days
    df['jail_time'] = df['jail_time'].fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    
    y = df['two_year_recid'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']
    
    numeric_cols = ['age','priors_count','days_b_screening_arrest','decile_score',
                    'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    categorical_cols = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['is_recid','two_year_recid','race_label','sex_label'])
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc(X_train_raw.index).reset_index(drop=True)
    bbn_test = bbn_df.loc(X_test_raw.index).reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached COMPAS data to {cache_file}")
    
    return result

def preprocess_german_for_fair_bbn(csv_path="/kaggle/input/all-datasets/german.data", seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_german.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached German data...")
        return joblib.load(cache_file)
    
    column_names = [
        "status", "duration", "credit_history", "purpose", "amount", "savings", "employment",
        "installment_rate", "personal_status_sex", "other_debtors", "residence", "property", "age",
        "other_installment_plans", "housing", "number_credits", "job", "people_liable",
        "telephone", "foreign_worker", "credit"
    ]
    df = pd.read_csv(csv_path, sep=' ', names=column_names)
    
    sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex'] = df['personal_status_sex'].map(sex_map)
    df['sex_label'] = df['sex'].map({'male':0,'female':1})
    df['age_label'] = (df['age'] >= 25).astype(int)
    df['foreign_worker_label'] = df['foreign_worker'].map({'A201':1,'A202':0})
    df['credit_label'] = df['credit'].map({1:1,2:0})
    df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
    
    X = df.drop(columns=['credit_label'])
    y = df['credit_label'].values
    sensitive_sex = df['sex_label'].values
    sensitive_age = df['age_label'].values
    sensitive_foreign = df['foreign_worker_label'].values
    
    numerical_cols = ["duration", "amount", "installment_rate", "residence","number_credits","people_liable"]
    categorical_cols = [col for col in X.columns if col not in numerical_cols]
    
    for col in numerical_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numerical_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['sex_label','age_label','foreign_worker_label']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X_train_raw, X_test_raw, y_train, y_test, sens_age_train, sens_age_test, sens_sex_train, sens_sex_test, sens_foreign_train, sens_foreign_test = train_test_split(
        X, y, sensitive_age, sensitive_sex, sensitive_foreign,
        test_size=0.2, random_state=seed, stratify=y
    )
    
    pipeline = Pipeline([('preprocessor', preproc)])
    X_train_nn = pipeline.fit_transform(X_train_raw)
    X_test_nn = pipeline.transform(X_test_raw)
    
    bbn_train_df = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test_df = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train_df, 'bbn_test_df': bbn_test_df,
        'y_train': y_train, 'y_test': y_test,
        'sens_age_train': sens_age_train, 'sens_age_test': sens_age_test,
        'sens_sex_train': sens_sex_train, 'sens_sex_test': sens_sex_test,
        'sens_foreign_train': sens_foreign_train, 'sens_foreign_test': sens_foreign_test
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached German data to {cache_file}")
    
    return result

def preprocess_lawschool_for_fair_bbn(law_path="/kaggle/input/all-datasets/bar_pass_prediction.csv", use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_lawschool.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Law School data...")
        return joblib.load(cache_file)
    
    law_df = pd.read_csv(law_path)
    law_df.columns = [c.strip().lower() for c in law_df.columns]
    target_col, sens_race, sens_gender = 'pass_bar', 'race', 'sex'
    law_df = law_df.dropna(subset=[target_col, sens_race, sens_gender])
    
    for col in law_df.select_dtypes(include=[np.number]).columns:
        law_df[col] = clean_numeric_column(law_df[col])
    
    law_df['income'] = np.where(law_df['fam_inc'] > law_df['fam_inc'].median(), 1, 0)
    
    law_df[sens_race] = law_df[sens_race].astype('category')
    law_df[sens_gender] = law_df[sens_gender].astype('category')
    law_df['race_label'] = law_df[sens_race].cat.codes
    law_df['gender_label'] = law_df[sens_gender].cat.codes
    
    numeric_cols = [c for c in ['lsat','ugpa','zfygpa','zgpa','age','gpa','fam_inc'] if c in law_df.columns]
    categorical_cols = [c for c in ['tier','cluster','fulltime','parttime'] if c in law_df.columns]
    
    low_var_cols = [col for col in numeric_cols if law_df[col].nunique() <= 1]
    if low_var_cols:
        law_df = law_df.drop(columns=low_var_cols)
        numeric_cols = [c for c in numeric_cols if c not in low_var_cols]
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = law_df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(law_df[col], 5)
    for col in categorical_cols + [sens_race, sens_gender]:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = law_df[numeric_cols + categorical_cols + [sens_race, sens_gender]]
    y = law_df['income'].values
    sens_race_labels = law_df['race_label']
    sens_gender_labels = law_df['gender_label']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race_labels, sens_gender_labels, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_gender_train': sens_gender_train.reset_index(drop=True),
        'sens_gender_test': sens_gender_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Law School data to {cache_file}")
    
    return result

def preprocess_diabetes_hospital_for_fair_bbn(csv_path='/kaggle/input/all-datasets/diabetes_hospital_fairlearn.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_hospital.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Hospital data...")
        return joblib.load(cache_file)
    
    df = pd.read_csv(csv_path)
    drop_cols = ["max_glu_serum", "A1Cresult", "readmitted.1"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df[~df.isin(['Missing']).any(axis=1)]
    df = df.dropna(subset=['race', 'gender']).reset_index(drop=True)
    
    age_map = {
        "'0-10'": 5, "'10-20'": 15, "'20-30'": 25, "'30-40'": 35, "'40-50'": 45,
        "'50-60'": 55, "'60-70'": 65, "'70-80'": 75, "'80-90'": 85, "'90-100'": 95,
        "'30 years or younger'": 20,
        "'30-60 years'": 45,
        "'Over 60 years'": 70
    }
    df['age'] = df['age'].replace(age_map).astype(float)
    df['readmit_binary'] = df['readmit_binary'].astype(int)
    df['race'] = df['race'].astype(str).str.strip()
    df['gender'] = df['gender'].astype(str).str.strip()
    
    race_map = {v: i for i, v in enumerate(df['race'].unique())}
    gender_map = {'Male': 0, 'Female': 1}
    df['race_label'] = df['race'].map(race_map)
    df['gender_label'] = df['gender'].map(gender_map)
    df['race_label'] = df['race_label'].fillna(0).astype(int)
    df['gender_label'] = df['gender_label'].fillna(0).astype(int)
    
    categorical_cols = [
        'discharge_disposition_id', 'admission_source_id',
        'medical_specialty', 'primary_diagnosis',
        'insulin', 'change', 'diabetesMed'
    ]
    numeric_cols = [
        'age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
        'num_medications', 'number_diagnoses', 'had_emergency',
        'had_inpatient_days', 'had_outpatient_days', 'medicare', 'medicaid'
    ]
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race', 'gender']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['readmit_binary', 'race_label', 'gender_label'])
    y = df['readmit_binary'].values
    sens_race = df['race_label']
    sens_gender = df['gender_label']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race, sens_gender, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn,
        'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train,
        'bbn_test_df': bbn_test,
        'y_train': y_train,
        'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_gender_train.reset_index(drop=True),
        'sens_sex_test': sens_gender_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Hospital data to {cache_file}")
    
    return result

def preprocess_bank_for_fair_bbn(csv_path='/kaggle/input/all-datasets/bank-full.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_bank.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Bank data...")
        return joblib.load(cache_file)
    
    try:
        df = pd.read_csv(csv_path, sep=';')
    except:
        df = pd.read_csv(csv_path, sep=',')
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    target_col = 'y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else 'subscribed'
    if target_col not in df.columns:
        target_col = df.columns[-1]
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y'] = np.where(df[target_col].isin(['yes', 'y', 'true', '1']), 1, 0)
    
    from sklearn.preprocessing import LabelEncoder
    le_marital = LabelEncoder()
    df['marital_label'] = le_marital.fit_transform(df['marital'])
    le_job = LabelEncoder()
    df['job_label'] = le_job.fit_transform(df['job'])
    
    categorical_cols = [col for col in ['job', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'] if col in df.columns]
    numeric_cols = [col for col in ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'] if col in df.columns]
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    for col in ['balance', 'duration', 'pdays', 'previous']:
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['marital', 'job']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['y', 'marital_label', 'job_label'])
    y = df['y'].values
    sens_marital = df['marital_label']
    sens_job = df['job_label']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_marital_train, sens_marital_test, sens_job_train, sens_job_test = \
        train_test_split(X, y, sens_marital, sens_job, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_marital_train': sens_marital_train.reset_index(drop=True),
        'sens_marital_test': sens_marital_test.reset_index(drop=True),
        'sens_job_train': sens_job_train.reset_index(drop=True),
        'sens_job_test': sens_job_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Bank data to {cache_file}")
    
    return result

class SimpleFairBBN:
    def __init__(self, feature_names=None, target_name='y'):
        self.feature_names = feature_names or []
        self.target_name = target_name
        self.model = None
        self.inference = None

    def fit(self, df_discrete, y, include_sensitive=True):
        data = df_discrete.copy().reset_index(drop=True)
        data[self.target_name] = y

        candidate_features = list(df_discrete.columns)
        selected = candidate_features[:min(8, len(candidate_features))]

        if include_sensitive:
            sens_cols = ['marital', 'job', 'race', 'sex', 'gender', 
                         'age_label', 'sex_label', 'foreign_worker_label']
            for sens in sens_cols:
                if sens in df_discrete.columns and sens not in selected:
                    selected.append(sens)

        edges = [(f, self.target_name) for f in selected]
        self.feature_names = selected

        self.model = DiscreteBayesianNetwork(edges)
        self.model.fit(data, estimator=BayesianEstimator, 
                       prior_type='BDeu', equivalent_sample_size=10)
        self.inference = VariableElimination(self.model)

    def predict_proba(self, df_discrete):
        Xdf = df_discrete.reset_index(drop=True)
        probs = []

        for _, row in Xdf.iterrows():
            evidence = {}
            used = 0
            for f in self.feature_names:
                if f in row and not pd.isna(row[f]) and used < 4:
                    try:
                        evidence[f] = int(row[f])
                        used += 1
                    except:
                        continue

            try:
                if evidence:
                    q = self.inference.query(variables=[self.target_name], evidence=evidence)
                else:
                    q = self.inference.query(variables=[self.target_name])
                p1 = q.values[1] if len(q.values) == 2 else 0.5
            except Exception:
                p1 = 0.5

            probs.append(p1)

        return np.array(probs)

class AdapterNN(nn.Module):
    def __init__(self, in_dim=3, hidden_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.out = nn.Linear(hidden_dim // 2, 1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x, return_features=False):
        h = self.dropout(self.act(self.fc1(x)))
        h2 = self.dropout(self.act(self.fc2(h)))
        logit = self.out(h2)
        return (logit, h2) if return_features else logit

class AdversaryNN(nn.Module):
    def __init__(self, in_dim, marital_classes, job_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Dropout(0.2))
        self.marital_head = nn.Linear(32, marital_classes)
        self.job_head = nn.Linear(32, job_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.marital_head(s), self.job_head(s)

class AdversaryNN_LawSchool(nn.Module):
    def __init__(self, in_dim, race_classes, gender_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Dropout(0.2))
        self.race_head = nn.Linear(32, race_classes)
        self.gender_head = nn.Linear(32, gender_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.gender_head(s)

class AdversaryNN_Adult(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Dropout(0.2))
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

class AdversaryNN_COMPAS(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Dropout(0.2))
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

class AdversaryNN_German(nn.Module):
    def __init__(self, in_dim, age_classes, sex_classes, foreign_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Dropout(0.2))
        self.age_head = nn.Linear(32, age_classes)
        self.sex_head = nn.Linear(32, sex_classes)
        self.foreign_head = nn.Linear(32, foreign_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.age_head(s), self.sex_head(s), self.foreign_head(s)

class AdversaryNN_Hospital(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Dropout(0.2))
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

def train_fair_bbn_adapter(data_dict, dataset_type, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3, hidden_dim=64, verbose=False):
    X_train_nn, X_test_nn = data_dict['X_train_nn'], data_dict['X_test_nn']
    bbn_train_df, bbn_test_df = data_dict['bbn_train_df'], data_dict['bbn_test_df']
    y_train, y_test = data_dict['y_train'], data_dict['y_test']

    config = {
        'german': {
            'sens_attrs': [('sens_age_train', 'sens_age_test'), ('sens_sex_train', 'sens_sex_test'), ('sens_foreign_train', 'sens_foreign_test')],
            'marginal_features': [['age_label'], ['sex_label'], ['foreign_worker_label']],
            'adversary_class': 'AdversaryNN_German',
            'adversary_kwargs': {'age_classes': 2, 'sex_classes': 2, 'foreign_classes': 2},
            'target_name': 'credit_label',
        },
        'compas': {
            'sens_attrs': [('sens_race_train', 'sens_race_test'), ('sens_sex_train', 'sens_sex_test')],
            'marginal_features': [['race'], ['sex']],
            'adversary_class': 'AdversaryNN_COMPAS',
            'target_name': 'two_year_recid',
        },
        'adult': {
            'sens_attrs': [('sens_race_train', 'sens_race_test'), ('sens_sex_train', 'sens_sex_test')],
            'marginal_features': [['race'], ['sex']],
            'adversary_class': 'AdversaryNN_Adult',
            'target_name': 'income',
        },
        'bank': {
            'sens_attrs': [('sens_marital_train', 'sens_marital_test'), ('sens_job_train', 'sens_job_test')],
            'marginal_features': [['marital'], ['job']],
            'adversary_class': 'AdversaryNN',
            'features_subset': ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous', 'marital', 'job'],
            'target_name': 'y',
        },
        'lawschool': {
            'sens_attrs': [('sens_race_train', 'sens_race_test'), ('sens_gender_train', 'sens_gender_test')],
            'marginal_features': [['race'], ['gender']],
            'adversary_class': 'AdversaryNN_LawSchool',
            'features_subset': ['lsat', 'ugpa', 'fulltime', 'fam_inc', 'age', 'race', 'gender'],
            'target_name': 'income',
        },
        'hospital': {
            'sens_attrs': [('sens_race_train', 'sens_race_test'), ('sens_sex_train', 'sens_sex_test')],
            'marginal_features': [['race'], ['gender']],
            'adversary_class': 'AdversaryNN_Hospital',
            'target_name': 'readmit_binary',
        },
    }

    cfg = config[dataset_type]
    
    if 'features_subset' in cfg:
        features = [f for f in cfg['features_subset'] if f in bbn_train_df.columns]
        bbn_train_sub = bbn_train_df[features]
        bbn_test_sub = bbn_test_df[features]
    else:
        bbn_train_sub = bbn_train_df
        bbn_test_sub = bbn_test_df

    if verbose:
        print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000, random_state=seed, early_stopping=True, learning_rate_init=0.001)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)

    base_results = {'pred': base_pred, 'acc': base_acc}
    for i, (train_attr, test_attr) in enumerate(cfg['sens_attrs']):
        sens_train = data_dict[train_attr]
        sens_test = data_dict[test_attr]
        attr_name = train_attr.replace('sens_', '').replace('_train', '')
        base_results[f'{attr_name}_dp'] = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_test)
        base_results[f'{attr_name}_eo'] = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_test)

    if verbose:
        print(f"Baseline MLP Accuracy: {base_acc:.4f}")
        print("Training fair BBN...")
        
    bbn = SimpleFairBBN(feature_names=list(bbn_train_sub.columns), target_name=cfg.get('target_name'))
    bbn.fit(bbn_train_sub, y_train, include_sensitive=True)

    p_all = bbn.predict_proba(bbn_train_sub).reshape(-1, 1)
    p_all_test = bbn.predict_proba(bbn_test_sub).reshape(-1, 1)
    
    marginals_train = [p_all]
    marginals_test = [p_all_test]
    
    for feat in cfg['marginal_features']:
        marginals_train.append(bbn.predict_proba(bbn_train_sub[feat]).reshape(-1, 1))
        marginals_test.append(bbn.predict_proba(bbn_test_sub[feat]).reshape(-1, 1))
    
    adapter_in_train = torch.tensor(np.hstack(marginals_train), dtype=torch.float32).to(device)
    adapter_in_test = torch.tensor(np.hstack(marginals_test), dtype=torch.float32).to(device)

    y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32).to(device)

    sens_tensors = []
    for train_attr, test_attr in cfg['sens_attrs']:
        sens_train = data_dict[train_attr]
        if hasattr(sens_train, 'values'):
            sens_train = sens_train.values
        sens_tensors.append(torch.tensor(sens_train.astype(int), dtype=torch.long).to(device))

    train_dataset = TensorDataset(adapter_in_train, y_train_t, *sens_tensors)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    adapter = AdapterNN(in_dim=len(marginals_train), hidden_dim=hidden_dim).to(device)
    
    adv_kwargs = {'in_dim': hidden_dim // 2}
    if dataset_type == 'german':
        adv_kwargs.update({'age_classes': 2, 'sex_classes': 2, 'foreign_classes': 2})
        adversary = AdversaryNN_German(**adv_kwargs).to(device)
    elif dataset_type == 'compas':
        adv_kwargs.update({'race_classes': len(np.unique(data_dict['sens_race_train'])), 'sex_classes': len(np.unique(data_dict['sens_sex_train']))})
        adversary = AdversaryNN_COMPAS(**adv_kwargs).to(device)
    elif dataset_type == 'adult':
        adv_kwargs.update({'race_classes': len(np.unique(data_dict['sens_race_train'])), 'sex_classes': len(np.unique(data_dict['sens_sex_train']))})
        adversary = AdversaryNN_Adult(**adv_kwargs).to(device)
    elif dataset_type == 'bank':
        adv_kwargs.update({'marital_classes': len(np.unique(data_dict['sens_marital_train'])), 'job_classes': len(np.unique(data_dict['sens_job_train']))})
        adversary = AdversaryNN(**adv_kwargs).to(device)
    elif dataset_type == 'lawschool':
        adv_kwargs.update({'race_classes': len(np.unique(data_dict['sens_race_train'])), 'gender_classes': len(np.unique(data_dict['sens_gender_train']))})
        adversary = AdversaryNN_LawSchool(**adv_kwargs).to(device)
    elif dataset_type == 'hospital':
        adv_kwargs.update({'race_classes': len(np.unique(data_dict['sens_race_train'])), 'sex_classes': len(np.unique(data_dict['sens_sex_train']))})
        adversary = AdversaryNN_Hospital(**adv_kwargs).to(device)

    adapter_opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=1e-5)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr * 1.5, weight_decay=1e-5)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    if verbose:
        print("Training adapter with adversarial + BBN fairness...")
    
    best_fair_score = float('inf')
    patience = 15
    no_improve = 0
    
    for epoch in range(epochs):
        adapter.train()
        adversary.train()
        total_adapter_loss = 0.0
        total_adv_loss = 0.0

        for batch in train_loader:
            batch_in, batch_y = batch[0], batch[1]
            batch_sens = batch[2:]

            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            
            if dataset_type == 'german':
                age_logits, sex_logits, foreign_logits = adversary(features)
                adv_loss = (adv_loss_fn(age_logits, batch_sens[0]) + adv_loss_fn(sex_logits, batch_sens[1]) + adv_loss_fn(foreign_logits, batch_sens[2])) / 3.0
            elif dataset_type == 'bank':
                m_logits, j_logits = adversary(features)
                adv_loss = (adv_loss_fn(m_logits, batch_sens[0]) + adv_loss_fn(j_logits, batch_sens[1])) / 2.0
            else:
                logits = adversary(features)
                adv_loss = sum(adv_loss_fn(logits[i], batch_sens[i]) for i in range(len(batch_sens))) / len(batch_sens)

            adv_loss.backward()
            torch.nn.utils.clip_grad_norm_(adversary.parameters(), 1.0)
            adv_opt.step()
            total_adv_loss += adv_loss.item()

            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit, batch_y)

            if dataset_type == 'german':
                age_logits2, sex_logits2, foreign_logits2 = adversary(features)
                adv_loss_for_adapter = (adv_loss_fn(age_logits2, batch_sens[0]) + adv_loss_fn(sex_logits2, batch_sens[1]) + adv_loss_fn(foreign_logits2, batch_sens[2])) / 3.0
                dp_penalties = []
                for sens_idx in range(3):
                    if (batch_sens[sens_idx]==0).sum() > 0 and (batch_sens[sens_idx]==1).sum() > 0:
                        dp_penalties.append(torch.abs(features[batch_sens[sens_idx]==0].mean() - features[batch_sens[sens_idx]==1].mean()))
                dp_penalty = torch.mean(torch.stack(dp_penalties)) if dp_penalties else torch.tensor(0.0).to(device)
            elif dataset_type == 'bank':
                m_logits2, j_logits2 = adversary(features)
                adv_loss_for_adapter = (adv_loss_fn(m_logits2, batch_sens[0]) + adv_loss_fn(j_logits2, batch_sens[1])) / 2.0
                dp_penalties = []
                for sens_idx in range(2):
                    uniq = torch.unique(batch_sens[sens_idx])
                    if len(uniq) > 1:
                        means = [features[batch_sens[sens_idx]==u].mean(dim=0) for u in uniq if (batch_sens[sens_idx]==u).sum() > 0]
                        if len(means) > 1:
                            for i in range(len(means)):
                                for j in range(i+1, len(means)):
                                    dp_penalties.append(torch.abs(means[i] - means[j]).mean())
                dp_penalty = torch.mean(torch.stack(dp_penalties)) if dp_penalties else torch.tensor(0.0).to(device)
            elif dataset_type == 'lawschool':
                logits2 = adversary(features)
                adv_loss_for_adapter = sum(adv_loss_fn(logits2[i], batch_sens[i]) for i in range(len(batch_sens))) / len(batch_sens)
                dp_penalties = []
                for sens_idx in range(len(batch_sens)):
                    with torch.no_grad():
                        pred_probs = torch.sigmoid(logit).detach()
                    uniq = torch.unique(batch_sens[sens_idx])
                    if len(uniq) > 1:
                        group_preds = {}
                        for u in uniq:
                            mask = batch_sens[sens_idx] == u
                            if mask.sum() > 0:
                                group_preds[u.item()] = pred_probs[mask].mean()
                        if len(group_preds) > 1:
                            vals = list(group_preds.values())
                            for i in range(len(vals)):
                                for j in range(i+1, len(vals)):
                                    dp_penalties.append(torch.abs(vals[i] - vals[j]))
                dp_penalty = torch.mean(torch.stack(dp_penalties)) if dp_penalties else torch.tensor(0.0).to(device)
            else:
                logits2 = adversary(features)
                adv_loss_for_adapter = sum(adv_loss_fn(logits2[i], batch_sens[i]) for i in range(len(batch_sens))) / len(batch_sens)
                dp_penalties = []
                for sens_idx in range(len(batch_sens)):
                    uniq = torch.unique(batch_sens[sens_idx])
                    if len(uniq) > 1:
                        means = [features[batch_sens[sens_idx]==u].mean(dim=0) for u in uniq if (batch_sens[sens_idx]==u).sum() > 0]
                        if len(means) > 1:
                            for i in range(len(means)):
                                for j in range(i+1, len(means)):
                                    dp_penalties.append(torch.abs(means[i] - means[j]).mean())
                dp_penalty = torch.mean(torch.stack(dp_penalties)) if dp_penalties else torch.tensor(0.0).to(device)

            total_loss = pred_loss - lambda_adv * adv_loss_for_adapter + alpha_bbn * dp_penalty
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
            adapter_opt.step()
            total_adapter_loss += total_loss.item()

        avg_adapter_loss = total_adapter_loss / len(train_loader)
        
        if epoch % 10 == 0:
            adapter.eval()
            with torch.no_grad():
                test_logit, _ = adapter(adapter_in_test, return_features=True)
                test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
                test_pred = (test_probs > 0.5).astype(int)
            
            fairness_metrics = []
            for i, (train_attr, test_attr) in enumerate(cfg['sens_attrs']):
                sens_test = data_dict[test_attr]
                dp_val = abs(demographic_parity_difference(y_test, test_pred, sensitive_features=sens_test))
                eo_val = abs(equalized_odds_difference(y_test, test_pred, sensitive_features=sens_test))
                fairness_metrics.extend([dp_val, eo_val])
            
            curr_fair_score = max(fairness_metrics) if fairness_metrics else 0
            if curr_fair_score < best_fair_score:
                best_fair_score = curr_fair_score
                no_improve = 0
            else:
                no_improve += 1
            
            if no_improve >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch}")
                break
            
            adapter.train()

        if verbose and (epoch % 20 == 0 or epoch == epochs - 1):
            print(f"Epoch {epoch:3d} | Adv: {total_adv_loss/len(train_loader):.4f} | Adapter: {avg_adapter_loss:.4f}")

    adapter.eval()
    adversary.eval()
    with torch.no_grad():
        test_logit, _ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs > 0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_results = {'pred': test_pred, 'acc': adv_acc}
    
    for i, (train_attr, test_attr) in enumerate(cfg['sens_attrs']):
        sens_test = data_dict[test_attr]
        attr_name = train_attr.replace('sens_', '').replace('_train', '')
        adv_results[f'{attr_name}_dp'] = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_test)
        adv_results[f'{attr_name}_eo'] = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_test)

    if verbose:
        print("\nBASELINE MLP RESULTS:")
        print(f" Accuracy: {base_acc:.4f}")
        for key, val in base_results.items():
            if key not in ['pred', 'acc']:
                print(f" {key}: {val:.4f}")

        print("\nBBN + Adapter RESULTS:")
        print(f" Accuracy: {adv_acc:.4f}")
        for key, val in adv_results.items():
            if key not in ['pred', 'acc']:
                print(f" {key}: {val:.4f}")

    return {'baseline': base_results, 'bbn_adapter': adv_results}

def optuna_objective(trial, data_dict, dataset_type):
    lambda_adv = trial.suggest_float('lambda_adv', 0.05, 2.0, log=True)
    alpha_bbn = trial.suggest_float('alpha_bbn', 0.05, 1.5, log=True)
    lr = trial.suggest_float('lr', 0.0001, 0.005, log=True)
    hidden_dim = trial.suggest_categorical('hidden_dim', [32, 64, 96, 128, 160])
    
    try:
        results = train_fair_bbn_adapter(
            data_dict, dataset_type,
            lambda_adv=lambda_adv,
            alpha_bbn=alpha_bbn,
            epochs=30,
            batch_size=256,
            lr=lr,
            hidden_dim=hidden_dim,
            verbose=False
        )
        
        acc = results['bbn_adapter']['acc']
        base_acc = results['baseline']['acc']
        
        fairness_metrics = [abs(v) for k, v in results['bbn_adapter'].items() 
                          if k not in ['pred', 'acc']]
        
        if not fairness_metrics:
            raise ValueError("No fairness metrics computed")
        
        max_fairness = max(fairness_metrics)
        avg_fairness = np.mean(fairness_metrics)
        acc_drop = max(0, base_acc - acc)
        
        score = (
            -10.0 * max_fairness
            -5.0 * avg_fairness
            -3.0 * acc_drop
            + 0.5 * acc
        )
        
        trial.report(max_fairness, step=0)
        
        trial.set_user_attr('accuracy', acc)
        trial.set_user_attr('baseline_accuracy', base_acc)
        trial.set_user_attr('accuracy_drop', acc_drop)
        trial.set_user_attr('max_fairness_violation', max_fairness)
        trial.set_user_attr('avg_fairness_violation', avg_fairness)
        
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        return score
        
    except Exception as e:
        print(f"Trial {trial.number} failed: {str(e)[:80]}")
        return -1000.0

def optimize_with_optuna(data_dict, dataset_type, n_trials=10, timeout=None):
    print(f"\n{'='*70}")
    print(f"OPTUNA OPTIMIZATION FOR {dataset_type.upper()}")
    print(f"Trials: {n_trials} | Timeout: {timeout}s" if timeout else f"Trials: {n_trials}")
    print(f"{'='*70}\n")
    
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(
            n_startup_trials=10,
            seed=42
        ),
        pruner=MedianPruner(
            n_startup_trials=5,
            n_warmup_steps=0
        ),
        study_name=f'fair_bbn_{dataset_type}'
    )
    
    study.optimize(
        lambda trial: optuna_objective(trial, data_dict, dataset_type),
        n_trials=n_trials,
        timeout=timeout,
        show_progress_bar=True,
        callbacks=[
            lambda study, trial: print(
                f"Trial {trial.number:3d} | "
                f"Score: {trial.value:7.4f} | "
                f"MaxFair: {trial.user_attrs.get('max_fairness_violation', float('nan')):.4f} | "
                f"Acc: {trial.user_attrs.get('accuracy', float('nan')):.4f} | "
                f"Drop: {trial.user_attrs.get('accuracy_drop', float('nan')):.4f}"
            ) if trial.value is not None else None
        ]
    )
    
    best_params = study.best_params
    
    print(f"\n{'='*70}")
    print(f"OPTIMIZATION COMPLETE FOR {dataset_type.upper()}")
    print(f"{'='*70}")
    print(f"Best Trial: #{study.best_trial.number}")
    print(f"Best Score: {study.best_value:.4f}")
    print(f"\nBest Hyperparameters:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")
    
    print(f"\nBest Trial Metrics:")
    print(f"  Accuracy: {study.best_trial.user_attrs['accuracy']:.4f}")
    print(f"  Baseline Accuracy: {study.best_trial.user_attrs['baseline_accuracy']:.4f}")
    print(f"  Accuracy Drop: {study.best_trial.user_attrs['accuracy_drop']:.4f}")
    print(f"  Max Fairness Violation: {study.best_trial.user_attrs['max_fairness_violation']:.4f}")
    print(f"  Avg Fairness Violation: {study.best_trial.user_attrs['avg_fairness_violation']:.4f}")
    
    pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
    complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    print(f"\nTrial Statistics:")
    print(f"  Complete: {len(complete)}")
    print(f"  Pruned: {len(pruned)}")
    print(f"  Failed: {n_trials - len(complete) - len(pruned)}")
    
    print(f"\n{'='*70}")
    print(f"TRAINING FINAL MODEL WITH BEST PARAMETERS")
    print(f"{'='*70}\n")
    
    final_results = train_fair_bbn_adapter(
        data_dict, dataset_type,
        lambda_adv=best_params['lambda_adv'],
        alpha_bbn=best_params['alpha_bbn'],
        epochs=80,
        batch_size=256,
        lr=best_params['lr'],
        hidden_dim=best_params['hidden_dim'],
        verbose=True
    )
    
    return best_params, final_results, study

if __name__ == '__main__':
    print("=" * 70)
    print("FAIR BBN SYSTEM WITH OPTUNA HYPERPARAMETER OPTIMIZATION")
    print("=" * 70)
    print(f"Device: {device}")
    print("=" * 70)

    datasets = [
        # ("ADULT INCOME", preprocess_adult_for_fair_bbn, "adult"),
        ("GERMAN CREDIT", preprocess_german_for_fair_bbn, "german"),
        ("COMPAS", preprocess_compas_for_fair_bbn, "compas"),
        ("BANK", preprocess_bank_for_fair_bbn, "bank"),
        ("LAWSCHOOL", preprocess_lawschool_for_fair_bbn, "lawschool"),
        ("DIABETES HOSPITAL", preprocess_diabetes_hospital_for_fair_bbn, "hospital"),
    ]

    all_results = {}
    all_best_params = {}
    all_studies = {}
    
    N_TRIALS = 10
    
    for name, preprocess_func, dataset_name in datasets:
        print(f"\n{'=' * 70}\n▶ PROCESSING {dataset_name.upper()} DATASET\n{'=' * 70}")
        
        try:
            data = preprocess_func(use_cache=True)
            
            best_params, results, study = optimize_with_optuna(
                data, 
                dataset_name, 
                n_trials=N_TRIALS
            )
            
            all_results[dataset_name] = results
            all_best_params[dataset_name] = best_params
            all_studies[dataset_name] = study
            
            print(f"✓ {name} completed successfully\n")
            
        except Exception as e:
            print(f"✗ {name} failed: {str(e)}\n")
            import traceback
            traceback.print_exc()
            continue

    print("\n" + "=" * 70)
    print("FINAL SUMMARY - ALL DATASETS")
    print("=" * 70)
    
    for dataset_name, results in all_results.items():
        print(f"\n{dataset_name.upper()}:")
        print(f"  Baseline Accuracy: {results['baseline']['acc']:.4f}")
        print(f"  BBN+Adapter Accuracy: {results['bbn_adapter']['acc']:.4f}")
        print(f"  Accuracy Drop: {(results['baseline']['acc'] - results['bbn_adapter']['acc']):.4f}")
        
        base_fairness = [abs(v) for k, v in results['baseline'].items() if k not in ['pred', 'acc']]
        bbn_fairness = [abs(v) for k, v in results['bbn_adapter'].items() if k not in ['pred', 'acc']]
        
        if base_fairness and bbn_fairness:
            print(f"  Baseline Max Fairness: {max(base_fairness):.4f}")
            print(f"  BBN+Adapter Max Fairness: {max(bbn_fairness):.4f}")
            print(f"  Fairness Improvement: {(max(base_fairness) - max(bbn_fairness)):.4f}")
        
        print(f"\n  Best Hyperparameters:")
        for param, value in all_best_params[dataset_name].items():
            print(f"    {param}: {value:.6f}" if isinstance(value, float) else f"    {param}: {value}")
        
        print(f"\n  Detailed Fairness Metrics:")
        print(f"  Baseline:")
        for k, v in results['baseline'].items():
            if k not in ['pred', 'acc']:
                print(f"    {k}: {v:.4f}")
        print(f"  BBN+Adapter:")
        for k, v in results['bbn_adapter'].items():
            if k not in ['pred', 'acc']:
                print(f"    {k}: {v:.4f}")
    
    print("\n" + "=" * 70)
    print("OPTIMIZED HYPERPARAMETERS (COPY FOR BEST_PARAMS)")
    print("=" * 70)
    print("BEST_PARAMS = {")
    for dataset_name, params in all_best_params.items():
        print(f"    '{dataset_name}': {params},")
    print("}")
    
    print("\n" + "=" * 70)
    print("✓ All datasets processed successfully!")
    print("=" * 70)
    
    try:
        import joblib
        for dataset_name, study in all_studies.items():
            study_file = f'/kaggle/working/optuna_study_{dataset_name}.pkl'
            joblib.dump(study, study_file)
            print(f"Saved study for {dataset_name} to {study_file}")
    except Exception as e:
        print(f"Could not save studies: {e}")

[I 2025-10-29 06:26:28,341] A new study created in memory with name: fair_bbn_german


Using device: cuda
FAIR BBN SYSTEM WITH OPTUNA HYPERPARAMETER OPTIMIZATION
Device: cuda

▶ PROCESSING GERMAN DATASET
Cached German data to /kaggle/working/preproc_german.pkl

OPTUNA OPTIMIZATION FOR GERMAN
Trials: 10



  0%|          | 0/10 [00:00<?, ?it/s]

[I 2025-10-29 06:33:43,056] Trial 0 finished with value: 0.2749999999999999 and parameters: {'lambda_adv': 0.19906996673933372, 'alpha_bbn': 1.2684995383408855, 'lr': 0.0017524101118128151, 'hidden_dim': 160}. Best is trial 0 with value: 0.2749999999999999.
Trial   0 | Score:  0.2750 | MaxFair: 0.0000 | Acc: 0.7000 | Drop: 0.0250
[I 2025-10-29 06:41:11,138] Trial 1 finished with value: 0.2749999999999999 and parameters: {'lambda_adv': 0.45918988705873276, 'alpha_bbn': 0.5557494721547487, 'lr': 0.00010838581269344756, 'hidden_dim': 32}. Best is trial 0 with value: 0.2749999999999999.
Trial   1 | Score:  0.2750 | MaxFair: 0.0000 | Acc: 0.7000 | Drop: 0.0250
[I 2025-10-29 06:48:22,916] Trial 2 finished with value: 0.2749999999999999 and parameters: {'lambda_adv': 0.15359756451337142, 'alpha_bbn': 0.2979194675034478, 'lr': 0.0005418282319533242, 'hidden_dim': 64}. Best is trial 0 with value: 0.2749999999999999.
Trial   2 | Score:  0.2750 | MaxFair: 0.0000 | Acc: 0.7000 | Drop: 0.0250
[I 20

Traceback (most recent call last):
  File "/tmp/ipykernel_19/2434763061.py", line 1095, in <cell line: 0>
    data = preprocess_func(use_cache=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_19/2434763061.py", line 177, in preprocess_compas_for_fair_bbn
    bbn_train = bbn_df.loc(X_train_raw.index).reset_index(drop=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pandas/core/indexing.py", line 738, in __call__
    axis_int_none = self.obj._get_axis_number(axis)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pandas/core/generic.py", line 576, in _get_axis_number
    return cls._AXIS_TO_AXIS_NUMBER[axis]
           ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
TypeError: unhashable type: 'Index'
[I 2025-10-29 07:44:03,691] A new study created in memory with name: fair_bbn_bank


...

[I 2025-10-29 08:01:53,239] Trial 0 finished with value: -10.898503706074184 and parameters: {'lambda_adv': 0.19906996673933372, 'alpha_bbn': 1.2684995383408855, 'lr': 0.0017524101118128151, 'hidden_dim': 160}. Best is trial 0 with value: -10.898503706074184.
Trial   0 | Score: -10.8985 | MaxFair: 1.0000 | Acc: 1.0000 | Drop: 0.0000
[I 2025-10-29 08:03:16,501] Trial 1 finished with value: -10.905463101873618 and parameters: {'lambda_adv': 0.45918988705873276, 'alpha_bbn': 0.5557494721547487, 'lr': 0.00010838581269344756, 'hidden_dim': 32}. Best is trial 0 with value: -10.898503706074184.
Trial   1 | Score: -10.9055 | MaxFair: 1.0000 | Acc: 0.9989 | Drop: 0.0011
[I 2025-10-29 08:04:40,198] Trial 2 finished with value: -10.898503706074184 and parameters: {'lambda_adv': 0.15359756451337142, 'alpha_bbn': 0.2979194675034478, 'lr': 0.0005418282319533242, 'hidden_dim': 64}. Best is trial 0 with value: -10.898503706074184.
Trial   2 | Score: -10.8985 | MaxFair: 1.0000 | Acc: 1.0000 | Drop: 0.0

[I 2025-10-29 08:16:44,287] A new study created in memory with name: fair_bbn_hospital


Cached Hospital data to /kaggle/working/preproc_hospital.pkl

OPTUNA OPTIMIZATION FOR HOSPITAL
Trials: 10



  0%|          | 0/10 [00:00<?, ?it/s]

[I 2025-10-29 08:20:41,108] Trial 0 finished with value: 0.06363373214974916 and parameters: {'lambda_adv': 0.19906996673933372, 'alpha_bbn': 1.2684995383408855, 'lr': 0.0017524101118128151, 'hidden_dim': 160}. Best is trial 0 with value: 0.06363373214974916.
Trial   0 | Score:  0.0636 | MaxFair: 0.0000 | Acc: 0.5580 | Drop: 0.0718
[I 2025-10-29 08:24:41,353] Trial 1 finished with value: 0.06363373214974916 and parameters: {'lambda_adv': 0.45918988705873276, 'alpha_bbn': 0.5557494721547487, 'lr': 0.00010838581269344756, 'hidden_dim': 32}. Best is trial 0 with value: 0.06363373214974916.
Trial   1 | Score:  0.0636 | MaxFair: 0.0000 | Acc: 0.5580 | Drop: 0.0718
[I 2025-10-29 08:28:38,756] Trial 2 finished with value: 0.06363373214974916 and parameters: {'lambda_adv': 0.15359756451337142, 'alpha_bbn': 0.2979194675034478, 'lr': 0.0005418282319533242, 'hidden_dim': 64}. Best is trial 0 with value: 0.06363373214974916.
Trial   2 | Score:  0.0636 | MaxFair: 0.0000 | Acc: 0.5580 | Drop: 0.0718